# exp083 R2 — convnext_pico × 5s Hi-freq mel × Multi-Iter Noisy Student (Colab Pro L4)

**exp083 R2 = R1 (val 0.9115、LB 0.886) + R1 pseudo distill**

## Changes from exp082 R2 (base)
- `TRAIN_DURATION` / `VAL_DURATION`: 20 → **5**
- Mel spec: Babych (224/4096/1252/0) → **Hi-freq (256/4096/512/20)**
  - 5s × 32000 / 512 = 313 frames
- `N_WINDOWS` in pseudo_infer: 3 → **12** (60s / 5s = 12 windows/file)
- LabeledSC aggregation: 4× 5s → 1× 5s = **no aggregation** (1:1)
- Perch distill: 4× 5s averaged → **1× 5s** (native 5s)
- BACKBONE: eca_nfnet_l0 → **convnext_pico.d1_in1k**
- Output: `output/exp083/r2/`, Kaggle slug `birdclef2026-exp083-weights`

## 不変項目
- Init: R1 ckpt warm start
- LR 3e-4 (ConvNeXt 適用)
- Batch: 96 (Colab L4 24GB、convnext_pico 軽量で余裕)
- N_TOTAL_EPOCHS 20 (R2 standard)
- ALPHA_DISTILL 1.0、Perch v2 distill
- SHARES focal 0.70 / labeled_sc 0.10 / pseudo_sc 0.20

## 想定時間 (Colab L4)
- DL data: 5-15 min (初回)
- R1 ckpt + pseudo CSV: Drive から copy ~1 min
- R2 train: 20 epoch × ~10-15 min/ep = ~3-5h
- Pseudo gen: ~10 min
- Drive mirror + Kaggle Dataset upload: 5-10 min
- **合計 ~4-6h、Colab L4 1 session 完走**

## 期待
- val_ns22: 0.93-0.94 (R1 0.9115 → R2 +0.02 lift)
- LB R2 single: 0.91-0.92
- ensemble bonus: Hi-freq mel diversity で +0.003-0.008


In [2]:
# ============================================================
# Cell 1: Setup — Drive mount, pip install, kaggle.json
# ============================================================
!pip install -q timm onnxruntime-gpu librosa soundfile scipy

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

import os, json, shutil, time, subprocess
from pathlib import Path

DRIVE_INPUT_DIR  = Path("/content/drive/MyDrive/kaggle/birdclef2026")
DRIVE_R1_PSEUDO  = DRIVE_INPUT_DIR / "output" / "exp083" / "r1-pseudo" / "pseudo_labels.csv"
DRIVE_OUTPUT_DIR = DRIVE_INPUT_DIR / "output" / "exp083" / "r2"
DRIVE_R2_PSEUDO_DIR = DRIVE_INPUT_DIR / "output" / "exp083" / "r2-pseudo"
DRIVE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DRIVE_R2_PSEUDO_DIR.mkdir(parents=True, exist_ok=True)
assert DRIVE_INPUT_DIR.exists(), f"Drive input folder missing: {DRIVE_INPUT_DIR}"
assert DRIVE_R1_PSEUDO.exists(), f"R1 pseudo CSV missing: {DRIVE_R1_PSEUDO}"
print(f"Drive input:    {DRIVE_INPUT_DIR}")
print(f"R1 pseudo CSV:  {DRIVE_R1_PSEUDO} ({DRIVE_R1_PSEUDO.stat().st_size/1e6:.1f}MB)")
print(f"Drive R2 ckpt:  {DRIVE_OUTPUT_DIR}")
print(f"Drive R2 pseudo:{DRIVE_R2_PSEUDO_DIR}")

# kaggle.json
KJ_CANDIDATES = [
    DRIVE_INPUT_DIR / "kaggle.json",
    Path("/content/drive/MyDrive/kaggle.json"),
]
KJ = next((p for p in KJ_CANDIDATES if p.exists()), None)
if KJ is not None:
    KAGGLE_CFG = Path.home() / ".kaggle"
    KAGGLE_CFG.mkdir(parents=True, exist_ok=True)
    shutil.copy(str(KJ), str(KAGGLE_CFG / "kaggle.json"))
    os.chmod(str(KAGGLE_CFG / "kaggle.json"), 0o600)
    creds = json.loads(KJ.read_text())
    if creds.get("key", "").startswith("KGAT_"):
        os.environ["KAGGLE_API_TOKEN"] = creds["key"]
    print(f"kaggle.json: {KJ}")
else:
    print("kaggle.json not found")

LOCAL_DATA = Path("/content/data")
LOCAL_OUT  = Path("/content/output")
LOCAL_DATA.mkdir(parents=True, exist_ok=True)
LOCAL_OUT.mkdir(parents=True, exist_ok=True)
print(f"Local data: {LOCAL_DATA}")
print(f"Local out:  {LOCAL_OUT}")


Mounted at /content/drive
Drive input:    /content/drive/MyDrive/kaggle/birdclef2026
R1 pseudo CSV:  /content/drive/MyDrive/kaggle/birdclef2026/output/exp083/r1-pseudo/pseudo_labels.csv (409.9MB)
Drive R2 ckpt:  /content/drive/MyDrive/kaggle/birdclef2026/output/exp083/r2
Drive R2 pseudo:/content/drive/MyDrive/kaggle/birdclef2026/output/exp083/r2-pseudo
kaggle.json: /content/drive/MyDrive/kaggle/birdclef2026/kaggle.json
Local data: /content/data
Local out:  /content/output


In [3]:
# ============================================================
# Cell 2: Data prep — Kaggle API direct DL + R1 pseudo copy (with tqdm)
# ============================================================
import time, zipfile, subprocess
from kaggle.api.kaggle_api_extended import KaggleApi
from tqdm.auto import tqdm

api = KaggleApi(); api.authenticate()
print("kaggle authenticated")

T0_total = time.time()

TA_DIR = LOCAL_DATA / "train_audio"
TS_DIR = LOCAL_DATA / "train_soundscapes"

need_dl = (
    not TA_DIR.exists() or sum(1 for _ in TA_DIR.rglob("*.ogg")) < 40000 or
    not TS_DIR.exists() or sum(1 for _ in TS_DIR.glob("*.ogg")) < 10000
)

if need_dl:
    print(f"\n[1/2] Downloading birdclef-2026 (~25GB) via Kaggle API...")
    t0 = time.time()
    api.competition_download_files(
        "birdclef-2026",
        path=str(LOCAL_DATA),
        force=False, quiet=False,
    )
    print(f"  DL done in {(time.time()-t0)/60:.1f} min")
    zips = list(LOCAL_DATA.glob("birdclef-2026*.zip"))
    assert zips
    zip_path = zips[0]
    print(f"\n[2/2] Extracting {zip_path.name} ({zip_path.stat().st_size/1e9:.1f}GB)...")
    t_extract = time.time()
    with zipfile.ZipFile(zip_path) as zf:
        infos = zf.infolist()
        total_bytes = sum(i.file_size for i in infos)
        pbar = tqdm(total=total_bytes, unit="B", unit_scale=True, unit_divisor=1024,
                    desc="extract", smoothing=0.05, mininterval=1.0)
        for info in infos:
            zf.extract(info, LOCAL_DATA)
            pbar.update(info.file_size)
        pbar.close()
    print(f"  extracted in {(time.time()-t_extract)/60:.1f} min")
    zip_path.unlink()
else:
    print("Competition data already present locally")

n_ta = sum(1 for _ in TA_DIR.rglob("*.ogg")) if TA_DIR.exists() else 0
n_ts = sum(1 for _ in TS_DIR.glob("*.ogg")) if TS_DIR.exists() else 0
print(f"\n  train_audio: {n_ta}, train_soundscapes: {n_ts}")

# Competition CSVs
comp = LOCAL_DATA / "competition"
comp.mkdir(parents=True, exist_ok=True)
for fn in ["train.csv", "taxonomy.csv", "sample_submission.csv", "train_soundscapes_labels.csv"]:
    src_in_root = LOCAL_DATA / fn
    dst = comp / fn
    if src_in_root.exists() and not dst.exists():
        shutil.copy2(str(src_in_root), str(dst))

# Perch ONNX
po_dir = LOCAL_DATA / "perch-onnx"
po_dir.mkdir(parents=True, exist_ok=True)
PERCH_PATH = po_dir / "perch_v2_no_dft.onnx"
if not PERCH_PATH.exists():
    drive_perch = list(DRIVE_INPUT_DIR.rglob("perch_v2*.onnx"))
    if drive_perch:
        src = drive_perch[0]; sz = src.stat().st_size
        print(f"\nCopying Perch ONNX from Drive ({sz/1e9:.2f}GB)...")
        with open(src, "rb") as fin, open(PERCH_PATH, "wb") as fout:
            pbar = tqdm(total=sz, unit="B", unit_scale=True, unit_divisor=1024,
                        desc="Perch copy", mininterval=1.0)
            while True:
                buf = fin.read(8*1024*1024)
                if not buf: break
                fout.write(buf); pbar.update(len(buf))
            pbar.close()
    else:
        print("\nPerch ONNX not on Drive, DL from Kaggle...")
        api.dataset_download_files("tuckerarrants/perch-v2-no-dft-onnx",
                                    path=str(po_dir), unzip=True, quiet=False)
        if not PERCH_PATH.exists():
            hits = list(po_dir.rglob("perch_v2*.onnx"))
            if hits: shutil.move(str(hits[0]), str(PERCH_PATH))
assert PERCH_PATH.exists()
print(f"Perch ONNX: {PERCH_PATH.stat().st_size/1e9:.2f}GB")

# R1 pseudo CSV (Drive → local for fast access)
LOCAL_PSEUDO_DIR = LOCAL_DATA / "r1-pseudo"
LOCAL_PSEUDO_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_PSEUDO_CSV = LOCAL_PSEUDO_DIR / "pseudo_labels.csv"
if not LOCAL_PSEUDO_CSV.exists():
    sz = DRIVE_R1_PSEUDO.stat().st_size
    print(f"\nCopying R1 pseudo CSV from Drive ({sz/1e6:.1f}MB)...")
    with open(DRIVE_R1_PSEUDO, "rb") as fin, open(LOCAL_PSEUDO_CSV, "wb") as fout:
        pbar = tqdm(total=sz, unit="B", unit_scale=True, unit_divisor=1024,
                    desc="pseudo copy", mininterval=1.0)
        while True:
            buf = fin.read(8*1024*1024)
            if not buf: break
            fout.write(buf); pbar.update(len(buf))
        pbar.close()
print(f"R1 pseudo CSV: {LOCAL_PSEUDO_CSV.stat().st_size/1e6:.1f}MB")

print(f"\n=== Total prep time: {(time.time()-T0_total)/60:.1f} min ===")
r = subprocess.run(["df", "-h", "/content"], capture_output=True, text=True)
print(r.stdout)


kaggle authenticated

[1/2] Downloading birdclef-2026 (~25GB) via Kaggle API...


100%|██████████| 15.0G/15.0G [06:29<00:00, 41.3MB/s]



  DL done in 6.5 min

[2/2] Extracting birdclef-2026.zip (16.1GB)...


extract:   0%|          | 0.00/15.0G [00:00<?, ?B/s]

  extracted in 1.0 min

  train_audio: 35549, train_soundscapes: 10658

Perch ONNX not on Drive, DL from Kaggle...
Dataset URL: https://www.kaggle.com/datasets/tuckerarrants/perch-v2-no-dft-onnx


100%|██████████| 431M/431M [00:11<00:00, 39.6MB/s]



Perch ONNX: 0.41GB

Copying R1 pseudo CSV from Drive (409.9MB)...


pseudo copy:   0%|          | 0.00/391M [00:00<?, ?B/s]

R1 pseudo CSV: 409.9MB

=== Total prep time: 8.9 min ===
Filesystem      Size  Used Avail Use% Mounted on
overlay         236G   65G  172G  28% /



In [4]:
# ============================================================
# Cell 3: Imports + Config + Paths (★ exp083: 5s Hi-freq mel + convnext_pico)
# ============================================================
import os, time, json, gc, random, math
from pathlib import Path
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, ConcatDataset
from torch.cuda.amp import GradScaler, autocast
import torchaudio
import timm
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold, GroupKFold
import warnings
warnings.filterwarnings("ignore")

# ===== Paths =====
BASE = LOCAL_DATA / "competition"
TA_DIR = LOCAL_DATA / "train_audio"
TS_DIR = LOCAL_DATA / "train_soundscapes"
TAXO_PATH = BASE / "taxonomy.csv"
TRAIN_CSV = BASE / "train.csv"
SAMPLE_SUB_PATH = BASE / "sample_submission.csv"
LABELS_PATH = BASE / "train_soundscapes_labels.csv"
PERCH_PATH = LOCAL_DATA / "perch-onnx" / "perch_v2_no_dft.onnx"
PSEUDO_CSV_PATH = LOCAL_DATA / "r1-pseudo" / "pseudo_labels.csv"
OUT_DIR = LOCAL_OUT
print(f"BASE: {BASE}")
print(f"TA_DIR: {TA_DIR.exists()}, TS_DIR: {TS_DIR.exists()}")
print(f"PERCH: {PERCH_PATH.exists()}, PSEUDO: {PSEUDO_CSV_PATH.exists()}")

# ===== Reproducibility =====
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)
torch.backends.cudnn.benchmark = True
torch.backends.cudnn.deterministic = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

# ===== Config =====
NUM_CLASSES = 234
SR = 32000
TRAIN_DURATION = 5    # ★ exp083: 20→5
VAL_DURATION   = 5    # ★ exp083: 20→5
TRAIN_SAMPLES  = SR * TRAIN_DURATION
VAL_SAMPLES    = SR * VAL_DURATION
N_FFT          = 4096   # ★ exp083: Hi-freq mel (R1 と同)
HOP_LENGTH     = 512    # ★ exp083: 1252→512 (5s で 313 frames)
N_MELS         = 256    # ★ exp083: 224→256
FMIN           = 20     # ★ exp083: 0→20
FMAX           = 16000
WIN_LENGTH     = N_FFT  # codebase convention

BACKBONE = "convnext_pico.d1_in1k"   # ★ exp083: convnext_pico

USE_PERCH_DISTILL = True
PERCH_EMBED_DIM = 1536
ALPHA_DISTILL = 1.0

N_FOLDS = 5
FOLDS = [0]
FORCE_FRESH = True       # ★ R2 = fresh init (warm start from R1 ckpt in cell 11)
# ★ exp083 R1 was BATCH=64 LR=3e-4 (Kaggle T4)。Colab L4 24GB なら BATCH=96 で sqrt scale LR
N_TOTAL_EPOCHS = 20      # R2 standard
BATCH = 96               # ★ convnext_pico 軽量、Colab L4 で余裕
LR = 3.7e-4              # ★ sqrt scaling (3e-4 × sqrt(96/64))、convnext は scale 適用可
MIN_LR = 1e-6
WD = 1e-4
WARMUP_EPOCHS = 3

# Augmentation
AUG_PROB = 0.5
AUG_GAIN_DB_RANGE = (-6.0, 6.0)
AUG_NOISE_SNR_DB_RANGE = (10.0, 30.0)

USE_MIXUP = True
MIXUP_PROB = 0.5
MIXUP_ALPHA = 0.4
MIXUP_HARD = False

FREQ_MASK_PARAM = 25
TIME_MASK_PARAM = 30
NUM_FREQ_MASKS = 2
NUM_TIME_MASKS = 2

MIN_SAMPLE = 20

# Source mixing (R2: 3 sources)
SHARES = {"focal": 0.70, "labeled_sc": 0.10, "pseudo_sc": 0.20}
SOURCE_WEIGHTS = {"focal": 1.0, "focal_missing": 0.0, "labeled_sc": 1.0, "pseudo_sc": 1.0}

NUM_WORKERS = 4
PERSISTENT_WORKERS = True

TRAIN_START = time.time()
MAX_RUNTIME_SEC = 22.0 * 3600

print(f"Backbone: {BACKBONE}")
print(f"Mel: {N_MELS} mels, n_fft={N_FFT}, hop={HOP_LENGTH}, win_length={WIN_LENGTH}")
print(f"Duration: {TRAIN_DURATION}s, samples={TRAIN_SAMPLES}")
print(f"Batch: {BATCH} | LR: {LR} | Epochs: {N_TOTAL_EPOCHS} | Folds: {FOLDS}")
print(f"SHARES: {SHARES}")


BASE: /content/data/competition
TA_DIR: True, TS_DIR: True
PERCH: True, PSEUDO: True
Device: cuda
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition
VRAM: 102.0 GB
Backbone: convnext_pico.d1_in1k
Mel: 256 mels, n_fft=4096, hop=512, win_length=4096
Duration: 5s, samples=160000
Batch: 96 | LR: 0.00037 | Epochs: 20 | Folds: [0]
SHARES: {'focal': 0.7, 'labeled_sc': 0.1, 'pseudo_sc': 0.2}


In [5]:
# ============================================================
# Cell 4: Load data — train.csv, taxonomy, labeled SS, R1 pseudo CSV
# ============================================================
sample_sub = pd.read_csv(SAMPLE_SUB_PATH)
PRIMARY_LABELS = sample_sub.columns[1:].tolist()
LABEL2IDX = {label: idx for idx, label in enumerate(PRIMARY_LABELS)}
assert len(PRIMARY_LABELS) == NUM_CLASSES

taxonomy = pd.read_csv(TAXO_PATH)
label_to_taxon = dict(zip(taxonomy["primary_label"].astype(str),
                          taxonomy["class_name"].astype(str)))
TAXON_MASKS = {t: np.array([i for i, l in enumerate(PRIMARY_LABELS)
                            if label_to_taxon.get(l, "") == t])
               for t in ["Aves", "Amphibia", "Insecta", "Mammalia", "Reptilia"]}

train_df = pd.read_csv(TRAIN_CSV)
train_df = train_df[train_df["primary_label"].astype(str).isin(LABEL2IDX)].reset_index(drop=True)
train_df["filename"] = train_df["filename"].astype(str)
print(f"Focal train.csv: {len(train_df)} rows")

def _check_exists(fn):
    return (TA_DIR / fn).exists()
print("Checking focal file existence...")
_t0 = time.time()
train_df["exists"] = train_df["filename"].map(_check_exists)
train_df = train_df[train_df["exists"]].drop(columns=["exists"]).reset_index(drop=True)
print(f"  {len(train_df)} focal files exist ({time.time()-_t0:.1f}s)")
train_df["original_idx"] = np.arange(len(train_df))

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
train_df["fold"] = -1
for fold, (_, val_idx) in enumerate(skf.split(train_df, train_df["primary_label"])):
    train_df.loc[val_idx, "fold"] = fold
print(f"Focal fold distribution: {train_df['fold'].value_counts().sort_index().to_dict()}")

focal_secondary_labels = {}
for idx, row in train_df.iterrows():
    sec = row.get("secondary_labels", "")
    if pd.isna(sec) or sec in ("", "[]"):
        continue
    try:
        sec_list = eval(sec) if isinstance(sec, str) else []
    except Exception:
        continue
    valid = [s for s in sec_list if s in LABEL2IDX]
    if valid:
        focal_secondary_labels[int(row["original_idx"])] = valid
print(f"Focal secondary labels: {len(focal_secondary_labels)} files")

counts = train_df["primary_label"].value_counts()
rare_species = counts[counts < MIN_SAMPLE].index.tolist()
extra_rows = []
for sp in rare_species:
    sp_rows = train_df[train_df["primary_label"] == sp]
    n_copies = int(np.ceil(MIN_SAMPLE / len(sp_rows))) - 1
    for _ in range(n_copies):
        extra_rows.append(sp_rows)
n_before = len(train_df)
if extra_rows:
    train_df = pd.concat([train_df] + extra_rows, ignore_index=True)
print(f"Upsampled {len(rare_species)} rare species (min={MIN_SAMPLE}): {n_before} -> {len(train_df)}")

# Labeled SS
if LABELS_PATH.exists():
    sc_labels_raw = pd.read_csv(LABELS_PATH).drop_duplicates()
    if sc_labels_raw["start"].dtype == object:
        sc_labels_raw["start_sec"] = pd.to_timedelta(sc_labels_raw["start"]).dt.total_seconds().astype(int)
    else:
        sc_labels_raw["start_sec"] = sc_labels_raw["start"].astype(int)

    sc_meta = (sc_labels_raw[["filename", "start_sec"]]
               .drop_duplicates()
               .reset_index(drop=True))
    if "site" in sc_labels_raw.columns:
        site_map = sc_labels_raw.groupby("filename")["site"].first().to_dict()
        sc_meta["site"] = sc_meta["filename"].map(site_map).fillna("UNK")
    else:
        sc_meta["site"] = "UNK"

    Y_SC = np.zeros((len(sc_meta), NUM_CLASSES), dtype=np.float32)
    for i, row in sc_meta.iterrows():
        matches = sc_labels_raw[(sc_labels_raw["filename"] == row["filename"]) &
                                 (sc_labels_raw["start_sec"] == row["start_sec"])]
        for _, m in matches.iterrows():
            for lbl in str(m["primary_label"]).split(";"):
                lbl = lbl.strip()
                if lbl in LABEL2IDX:
                    Y_SC[i, LABEL2IDX[lbl]] = 1.0
    print(f"Soundscape labels: {len(sc_meta)} windows, {int(Y_SC.sum())} positives, "
          f"{int((Y_SC.sum(axis=0) > 0).sum())} species")

    # ============================================================
    # ★ exp083: 5s native = no LabeledSC aggregation (1:1)
    sc_files = sc_meta[["filename", "site"]].drop_duplicates().reset_index(drop=True)
    gkf = GroupKFold(n_splits=N_FOLDS)
    sc_files["fold"] = -1
    for fold, (_, val_idx) in enumerate(gkf.split(sc_files, groups=sc_files["filename"])):
        sc_files.loc[sc_files.index[val_idx], "fold"] = fold
    file_to_fold = dict(zip(sc_files["filename"], sc_files["fold"]))
    sc_meta["fold"] = sc_meta["filename"].map(file_to_fold).fillna(-1).astype(int)
    non_s22_mask_sc = (sc_meta["site"].values != "S22")
    print(f"Non-S22: {non_s22_mask_sc.sum()}/{len(sc_meta)}")
else:
    print("LABELS_PATH missing")
    sc_meta = pd.DataFrame(columns=["filename", "start_sec", "site", "fold"])
    Y_SC = np.zeros((0, NUM_CLASSES), dtype=np.float32)
    non_s22_mask_sc = np.zeros(0, dtype=bool)

# ★ R1 pseudo CSV (R2 の 3rd source)
print(f"\nLoading R1 pseudo CSV: {PSEUDO_CSV_PATH}")
_t0 = time.time()
pseudo_df_full = pd.read_csv(PSEUDO_CSV_PATH)
print(f"  loaded {len(pseudo_df_full)} rows in {time.time()-_t0:.1f}s")

_label_cols_in_csv = [c for c in pseudo_df_full.columns if c in LABEL2IDX]
assert len(_label_cols_in_csv) == NUM_CLASSES, (
    f"pseudo CSV must contain all {NUM_CLASSES} species cols; got {len(_label_cols_in_csv)}"
)
Y_PSEUDO = pseudo_df_full[PRIMARY_LABELS].values.astype(np.float32)
pseudo_meta = pseudo_df_full[["filename", "start_sec"]].copy().reset_index(drop=True)
pseudo_meta["filename"] = pseudo_meta["filename"].astype(str)
pseudo_meta["start_sec"] = pseudo_meta["start_sec"].astype(float)
print(f"  Y_pseudo: {Y_PSEUDO.shape}, range=[{Y_PSEUDO.min():.4f}, {Y_PSEUDO.max():.4f}], "
      f"mean={Y_PSEUDO.mean():.4f}")
print(f"  pseudo_meta: {len(pseudo_meta)} rows, {pseudo_meta['filename'].nunique()} files")

print("\nOK data loaded")

# ============================================================
# ★ exp081: Verify pseudo is 10s native (from exp081 R1 NB output)
# ============================================================
# Expected: 6 rows per file (start_sec 0, 10, 20, 30, 40, 50)
# If pseudo CSV is from exp017 R1 (12 rows/file, 5s), need aggregation - WARN
_n_per_file = pseudo_meta.groupby("filename").size()
print(f"[exp081] Pseudo rows per file: mean={_n_per_file.mean():.1f}, median={_n_per_file.median():.0f}")
if _n_per_file.median() == 3:
    print("  [exp083] OK: 20s native pseudo (3 rows/file from exp083 R1)")
elif _n_per_file.median() == 6:
    print("  [exp083] WARN: 10s pseudo detected (6 rows/file). Aggregating to 20s...")
    import numpy as _np_e82
    import pandas as _pd_e82
    _psd_full = pseudo_meta.copy()
    _psd_full["_orig_idx"] = _np_e82.arange(len(_psd_full))
    _psd_full = _psd_full.sort_values(["filename", "start_sec"]).reset_index(drop=True)
    new_meta_rows = []
    new_Y_20s = []
    for fname, group in _psd_full.groupby("filename"):
        group_sorted = group.reset_index(drop=True)
        for k in range(0, len(group_sorted) - 1, 2):
            idx_a = int(group_sorted.iloc[k]["_orig_idx"])
            idx_b = int(group_sorted.iloc[k+1]["_orig_idx"])
            start_sec_20s = float(group_sorted.iloc[k]["start_sec"])
            y_agg = _np_e82.maximum(Y_PSEUDO[idx_a], Y_PSEUDO[idx_b])
            new_meta_rows.append({"filename": fname, "start_sec": start_sec_20s})
            new_Y_20s.append(y_agg)
    pseudo_meta = _pd_e82.DataFrame(new_meta_rows).reset_index(drop=True)
    Y_PSEUDO = _np_e82.array(new_Y_20s, dtype=_np_e82.float32)
    print(f"  After 10s -> 20s aggregation: {len(pseudo_meta)} rows")
elif _n_per_file.median() == 12:
    print("  [exp083] WARN: 5s pseudo detected (12 rows/file). Aggregating to 20s (4x)...")
    import numpy as _np_e82_b
    import pandas as _pd_e82_b
    _psd_full = pseudo_meta.copy()
    _psd_full["_orig_idx"] = _np_e82_b.arange(len(_psd_full))
    _psd_full = _psd_full.sort_values(["filename", "start_sec"]).reset_index(drop=True)
    new_meta_rows = []
    new_Y_20s = []
    for fname, group in _psd_full.groupby("filename"):
        group_sorted = group.reset_index(drop=True)
        # 12 5s rows -> 3 20s rows (4x aggregation)
        for k in range(0, len(group_sorted) - 3, 4):
            idx_a = int(group_sorted.iloc[k]["_orig_idx"])
            idx_b = int(group_sorted.iloc[k+1]["_orig_idx"])
            idx_c = int(group_sorted.iloc[k+2]["_orig_idx"])
            idx_d = int(group_sorted.iloc[k+3]["_orig_idx"])
            start_sec_20s = float(group_sorted.iloc[k]["start_sec"])
            y_agg = _np_e82_b.maximum.reduce([Y_PSEUDO[idx_a], Y_PSEUDO[idx_b],
                                              Y_PSEUDO[idx_c], Y_PSEUDO[idx_d]])
            new_meta_rows.append({"filename": fname, "start_sec": start_sec_20s})
            new_Y_20s.append(y_agg)
    pseudo_meta = _pd_e82_b.DataFrame(new_meta_rows).reset_index(drop=True)
    Y_PSEUDO = _np_e82_b.array(new_Y_20s, dtype=_np_e82_b.float32)
    print(f"  After 5s -> 20s aggregation: {len(pseudo_meta)} rows, Y shape {Y_PSEUDO.shape}")
else:
    print(f"  [exp081] WARN: unexpected pseudo row count {_n_per_file.median()}/file")
print(f"  [exp081] Y_PSEUDO mean: {Y_PSEUDO.mean():.5f}, max: {Y_PSEUDO.max():.4f}")

Focal train.csv: 35549 rows
Checking focal file existence...
  35549 focal files exist (0.2s)
Focal fold distribution: {0: 7110, 1: 7110, 2: 7110, 3: 7110, 4: 7109}
Focal secondary labels: 4372 files
Upsampled 36 rare species (min=20): 35549 -> 36135
Soundscape labels: 739 windows, 3122 positives, 75 species
Non-S22: 739/739

Loading R1 pseudo CSV: /content/data/r1-pseudo/pseudo_labels.csv
  loaded 127896 rows in 1.3s
  Y_pseudo: (127896, 234), range=[0.0000, 0.9990], mean=0.0059
  pseudo_meta: 127896 rows, 10658 files

OK data loaded
[exp081] Pseudo rows per file: mean=12.0, median=12
  [exp083] WARN: 5s pseudo detected (12 rows/file). Aggregating to 20s (4x)...
  After 5s -> 20s aggregation: 31974 rows, Y shape (31974, 234)
  [exp081] Y_PSEUDO mean: 0.00756, max: 0.9990


In [6]:
# ============================================================
# Cell 5: Model — Mel + SpecAugment + Perch teacher + DistillHead + BirdSEDModel
# ============================================================
import onnxruntime as ort

class MelSpecTransform(nn.Module):
    def __init__(self):
        super().__init__()
        self.mel_spec = torchaudio.transforms.MelSpectrogram(
            sample_rate=SR, n_fft=N_FFT, hop_length=HOP_LENGTH,
            n_mels=N_MELS, f_min=FMIN, f_max=FMAX, power=2.0,
        )
        self.db_transform = torchaudio.transforms.AmplitudeToDB(top_db=80)
    def forward(self, waveform):
        return self.db_transform(self.mel_spec(waveform))

class SpecAugment(nn.Module):
    def __init__(self):
        super().__init__()
        self.freq_mask = torchaudio.transforms.FrequencyMasking(freq_mask_param=FREQ_MASK_PARAM)
        self.time_mask = torchaudio.transforms.TimeMasking(time_mask_param=TIME_MASK_PARAM)
    def forward(self, mel):
        for _ in range(NUM_FREQ_MASKS): mel = self.freq_mask(mel)
        for _ in range(NUM_TIME_MASKS): mel = self.time_mask(mel)
        return mel

class PerchTeacher:
    def __init__(self, onnx_path, device_str="cuda"):
        providers = ["CUDAExecutionProvider", "CPUExecutionProvider"] \
            if device_str == "cuda" else ["CPUExecutionProvider"]
        self.session = ort.InferenceSession(str(onnx_path), providers=providers)
        self.input_name = self.session.get_inputs()[0].name
        self._embed_idx = None
        for i, o in enumerate(self.session.get_outputs()):
            if o.shape and o.shape[-1] == PERCH_EMBED_DIM:
                self._embed_idx = i; break
        if self._embed_idx is None:
            self._embed_idx = 1
        print(f"PerchTeacher: embed_idx={self._embed_idx}, providers={self.session.get_providers()}")

    @torch.no_grad()
    def embed(self, waveforms_5s):
        wav_np = waveforms_5s.cpu().numpy().astype(np.float32)
        results = self.session.run(None, {self.input_name: wav_np})
        return torch.from_numpy(results[self._embed_idx]).float()

class DistillHead(nn.Module):
    def __init__(self, backbone_dim, embed_dim=1536):
        super().__init__()
        self.proj = nn.Linear(backbone_dim, embed_dim)
    def forward(self, feature_map):
        return self.proj(feature_map.mean(dim=[2, 3]))

class GeMFreqPool(nn.Module):
    def __init__(self, p_init=3.0, eps=1e-6):
        super().__init__()
        self.p = nn.Parameter(torch.tensor(float(p_init)))
        self.eps = eps
    def forward(self, x):
        p = self.p.clamp(min=1.0)
        x = x.clamp(min=self.eps).pow(p)
        x = x.mean(dim=2)
        return x.pow(1.0 / p)

class BirdSEDModel(nn.Module):
    def __init__(self, backbone_name=BACKBONE, num_classes=NUM_CLASSES,
                 drop_path_rate=0.1, hidden_dim=512):
        super().__init__()
        self.backbone = timm.create_model(
            backbone_name, pretrained=True, in_chans=1,
            num_classes=0, global_pool="", drop_path_rate=drop_path_rate,
        )
        with torch.no_grad():
            n_tf = TRAIN_SAMPLES // HOP_LENGTH + 1
            dummy = torch.randn(1, 1, N_MELS, n_tf)
            feat = self.backbone(dummy)
            self.backbone_dim = feat.shape[1]
            print(f"Backbone out: {tuple(feat.shape)}  (C={self.backbone_dim})")

        self.gem_freq = GeMFreqPool(p_init=3.0)
        self.dense = nn.Sequential(
            nn.Dropout(0.25),
            nn.Linear(self.backbone_dim, hidden_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
        )
        self.att = nn.Conv1d(hidden_dim, num_classes, kernel_size=1, bias=True)
        self.cla = nn.Conv1d(hidden_dim, num_classes, kernel_size=1, bias=True)
        nn.init.xavier_uniform_(self.att.weight)
        nn.init.xavier_uniform_(self.cla.weight)
        self.att.bias.data.fill_(0.)
        self.cla.bias.data.fill_(0.)
        if USE_PERCH_DISTILL:
            self.distill_head = DistillHead(self.backbone_dim, PERCH_EMBED_DIM)

    def forward(self, x, return_framewise=False, return_distill=False):
        h = self.backbone(x)
        distill_emb = None
        if return_distill and hasattr(self, "distill_head"):
            distill_emb = self.distill_head(h)
        h_cls = h.detach() if USE_PERCH_DISTILL else h
        h_cls = self.gem_freq(h_cls)
        h_cls = h_cls.permute(0, 2, 1)
        h_cls = self.dense(h_cls)
        h_cls = h_cls.permute(0, 2, 1)
        norm_att = torch.softmax(torch.tanh(self.att(h_cls)), dim=-1)
        framewise_logits = self.cla(h_cls)
        clip_logits = torch.sum(norm_att * framewise_logits, dim=2)
        fw = framewise_logits.permute(0, 2, 1) if return_framewise else None
        if return_framewise and return_distill:
            return clip_logits, fw, distill_emb
        elif return_framewise:
            return clip_logits, fw
        elif return_distill:
            return clip_logits, distill_emb
        return clip_logits

def make_model():
    m = BirdSEDModel().to(device)
    m = m.to(memory_format=torch.channels_last)
    return m

print("OK model defs ready")


OK model defs ready


In [7]:
# ============================================================
# Cell 6: Datasets — FocalDS, LabeledSCDS, PseudoScDS, samplers
# ============================================================
import soundfile as sf
import librosa
from functools import lru_cache

# LRU cache for full 60s SS waveforms (12 windows × 5s same file → cache hit)
# ★ workers=8 で並列度上げる時、cache 容量も増やす (per-worker、各 ~4GB RAM)
@lru_cache(maxsize=512)
def _load_full_audio_cached(path_str):
    try:
        wav, sr = sf.read(path_str, dtype="float32", always_2d=False)
        if wav.ndim > 1:
            wav = wav.mean(axis=1)
        if sr != SR:
            wav = librosa.resample(wav, orig_sr=sr, target_sr=SR)
        return wav.astype(np.float32)
    except Exception:
        return None

def _load_ogg(path, n_samples_target, start_sec=None):
    if start_sec is not None:
        full = _load_full_audio_cached(str(path))
        if full is None:
            return None
        start = int(start_sec * SR)
        end = start + n_samples_target
        if end <= len(full):
            return full[start:end].copy()
        out = np.zeros(n_samples_target, dtype=np.float32)
        avail = full[start:start + n_samples_target]
        out[:len(avail)] = avail
        return out
    try:
        wav, sr = sf.read(str(path), dtype="float32", always_2d=False)
        if wav.ndim > 1:
            wav = wav.mean(axis=1)
        if sr != SR:
            wav = librosa.resample(wav, orig_sr=sr, target_sr=SR)
        return wav.astype(np.float32)
    except Exception:
        return None

def extract_chunk_np(waveform, start_sample, n_samples):
    total = len(waveform)
    if total <= n_samples:
        return np.pad(waveform, (n_samples - total, 0))
    end = start_sample + n_samples
    if end > total:
        start_sample = max(0, total - n_samples)
    return waveform[start_sample:start_sample + n_samples]

def apply_aug(w):
    if np.random.random() < AUG_PROB:
        w = w * (10 ** (np.random.uniform(*AUG_GAIN_DB_RANGE) / 20))
    if np.random.random() < AUG_PROB:
        sp = (w ** 2).mean()
        if sp > 1e-10:
            w = w + np.random.randn(*w.shape).astype(w.dtype) * np.sqrt(
                sp / (10 ** (np.random.uniform(*AUG_NOISE_SNR_DB_RANGE) / 10)))
    return w


class FocalDS(Dataset):
    def __init__(self, df, l2i, secondary_lookup=None, aug=False):
        self.df = df.reset_index(drop=True)
        self.l2i = l2i
        self.aug = aug
        self.secondary_lookup = secondary_lookup
        self.filenames = self.df["filename"].values
        self.primary = self.df["primary_label"].astype(str).values
        self.original_idx = self.df["original_idx"].values if "original_idx" in self.df.columns else None

    def __len__(self): return len(self.df)

    def _load_chunk(self, i):
        fn = self.filenames[i]
        path = TA_DIR / fn
        wav = _load_ogg(path, n_samples_target=None)
        if wav is None:
            return None, None
        if self.aug and len(wav) > TRAIN_SAMPLES:
            start = np.random.randint(0, len(wav) - TRAIN_SAMPLES + 1)
        else:
            start = 0
        chunk = extract_chunk_np(wav, start, TRAIN_SAMPLES)
        lb = np.zeros(NUM_CLASSES, dtype=np.float32)
        if self.primary[i] in self.l2i:
            lb[self.l2i[self.primary[i]]] = 1.0
        if self.secondary_lookup is not None and self.original_idx is not None:
            for s in self.secondary_lookup.get(int(self.original_idx[i]), []):
                if s in self.l2i: lb[self.l2i[s]] = 1.0
        return chunk, lb

    def __getitem__(self, i):
        ch1, lb1 = self._load_chunk(i)
        if ch1 is None:
            return (torch.zeros(1, TRAIN_SAMPLES), torch.zeros(NUM_CLASSES),
                    torch.ones(NUM_CLASSES), torch.ones(NUM_CLASSES), "focal_missing")

        if USE_MIXUP and self.aug and np.random.random() < MIXUP_PROB:
            ch2, lb2 = None, None
            for _ in range(3):
                j = np.random.randint(len(self.df))
                ch2, lb2 = self._load_chunk(j)
                if ch2 is not None: break
            if ch2 is not None:
                lam = np.random.beta(MIXUP_ALPHA, MIXUP_ALPHA)
                ch_mix = (lam * ch1 + (1 - lam) * ch2).astype(np.float32)
                if self.aug: ch_mix = apply_aug(ch_mix)
                lb = np.maximum(lb1, lb2) if MIXUP_HARD else (lam * lb1 + (1 - lam) * lb2)
                return (torch.from_numpy(ch_mix).unsqueeze(0),
                        torch.from_numpy(lb.astype(np.float32)),
                        torch.ones(NUM_CLASSES), torch.ones(NUM_CLASSES), "focal")

        if self.aug: ch1 = apply_aug(ch1)
        return (torch.from_numpy(ch1.astype(np.float32)).unsqueeze(0),
                torch.from_numpy(lb1),
                torch.ones(NUM_CLASSES), torch.ones(NUM_CLASSES), "focal")


class LabeledSCDS(Dataset):
    def __init__(self, Y, sc_df, aug=False):
        self.Y = Y
        self.df = sc_df.reset_index(drop=True)
        self.aug = aug
        self.filenames = self.df["filename"].astype(str).values
        self.start_secs = self.df["start_sec"].astype(int).values

    def __len__(self): return len(self.df)

    def __getitem__(self, i):
        fn = self.filenames[i]
        if not fn.endswith(".ogg"): fn = fn + ".ogg"
        path = TS_DIR / fn
        wav = _load_ogg(path, n_samples_target=TRAIN_SAMPLES,
                         start_sec=float(self.start_secs[i]))
        if wav is None:
            wav = np.zeros(TRAIN_SAMPLES, dtype=np.float32)
        if len(wav) < TRAIN_SAMPLES:
            wav = np.pad(wav, (0, TRAIN_SAMPLES - len(wav)))
        else:
            wav = wav[:TRAIN_SAMPLES]
        if self.aug:
            wav = apply_aug(wav)
        return (torch.from_numpy(wav.astype(np.float32)).unsqueeze(0),
                torch.from_numpy(self.Y[i].astype(np.float32)),
                torch.ones(NUM_CLASSES), torch.ones(NUM_CLASSES), "labeled_sc")


class PseudoScDS(Dataset):
    '''R1 pseudo-labeled SS windows. Soft labels.'''
    def __init__(self, meta_df, Y_soft, audio_dir, aug=False):
        self.meta = meta_df.reset_index(drop=True)
        self.Y = Y_soft.astype(np.float32)
        self.audio_dir = Path(audio_dir)
        self.aug = aug
        self.filenames = self.meta["filename"].values
        self.start_secs = self.meta["start_sec"].values

    def __len__(self): return len(self.meta)

    def __getitem__(self, i):
        fn = str(self.filenames[i])
        if not fn.endswith(".ogg"):
            fn = fn + ".ogg"
        path = self.audio_dir / fn
        start_sec = float(self.start_secs[i])
        wav = _load_ogg(path, n_samples_target=TRAIN_SAMPLES, start_sec=start_sec)
        if wav is None:
            wav = np.zeros(TRAIN_SAMPLES, dtype=np.float32)
        if len(wav) < TRAIN_SAMPLES:
            wav = np.pad(wav, (0, TRAIN_SAMPLES - len(wav)))
        else:
            wav = wav[:TRAIN_SAMPLES]
        if self.aug:
            wav = apply_aug(wav)
        return (torch.from_numpy(wav.astype(np.float32)).unsqueeze(0),
                torch.from_numpy(self.Y[i]),
                torch.ones(NUM_CLASSES), torch.ones(NUM_CLASSES), "pseudo_sc")


class MixSamp(torch.utils.data.Sampler):
    def __init__(self, sizes, names, shares, bs, nst, seed=0):
        self.sizes, self.names, self.bs, self.nst = sizes, names, bs, nst
        self.rng = np.random.default_rng(seed)
        per_src = [max(1, int(round(bs * shares.get(n, 0.0)))) for n in names]
        total = sum(per_src)
        if total != bs:
            per_src[int(np.argmax(per_src))] += (bs - total)
        self.per_src = per_src
        self.offsets = [0]
        for s in sizes[:-1]:
            self.offsets.append(self.offsets[-1] + s)
    def __len__(self): return self.nst
    def __iter__(self):
        for _ in range(self.nst):
            batch = []
            for off, size, n in zip(self.offsets, self.sizes, self.per_src):
                if n <= 0 or size <= 0: continue
                idxs = self.rng.integers(0, size, size=n)
                batch.extend([off + int(i) for i in idxs])
            self.rng.shuffle(batch)
            yield batch

def collate_m(batch):
    return (torch.stack([b[0] for b in batch]),
            torch.stack([b[1] for b in batch]),
            torch.stack([b[2] for b in batch]),
            torch.stack([b[3] for b in batch]),
            [b[4] for b in batch])

def mk_sw(sr):
    return torch.tensor([SOURCE_WEIGHTS.get(s, 0.0) for s in sr], dtype=torch.float32)

print("OK datasets ready (focal + labeled_sc + pseudo_sc)")


OK datasets ready (focal + labeled_sc + pseudo_sc)


In [8]:
# ============================================================
# Cell 7: Eval — AUC + validation predictor
# ============================================================
def compute_macro_auc(y_true, y_pred, mask=None, class_mask=None):
    if mask is not None:
        y_true, y_pred = y_true[mask], y_pred[mask]
    if class_mask is not None:
        y_true, y_pred = y_true[:, class_mask], y_pred[:, class_mask]
    aucs = []
    for c in range(y_true.shape[1]):
        col = y_true[:, c]
        if col.sum() == 0 or col.sum() == len(col):
            continue
        try:
            aucs.append(roc_auc_score(col, y_pred[:, c]))
        except ValueError:
            continue
    return (np.mean(aucs) if aucs else float("nan")), len(aucs)


def full_eval(y_true, y_pred, ns22_mask, taxon_masks):
    r = {}
    a, n = compute_macro_auc(y_true, y_pred)
    r["macro_auc_all"] = round(float(a) if not np.isnan(a) else 0.0, 4)
    r["n_all"] = n
    a, n = compute_macro_auc(y_true, y_pred, mask=ns22_mask)
    r["non_s22_macro"] = round(float(a) if not np.isnan(a) else 0.0, 4)
    r["n_ns22"] = n
    per_taxon = {}
    for t, cm in taxon_masks.items():
        a, n = compute_macro_auc(y_true, y_pred, mask=ns22_mask, class_mask=cm)
        per_taxon[t] = round(float(a) if not np.isnan(a) else 0.0, 4)
    r["per_taxon"] = per_taxon
    return r


def _load_val_waveforms(val_sc_df):
    wavs = []
    for _, row in val_sc_df.iterrows():
        fn = str(row["filename"])
        if not fn.endswith(".ogg"): fn = fn + ".ogg"
        wav = _load_ogg(TS_DIR / fn, n_samples_target=VAL_SAMPLES,
                         start_sec=float(row["start_sec"]))
        if wav is None or len(wav) == 0:
            wav = np.zeros(VAL_SAMPLES, dtype=np.float32)
        if len(wav) < VAL_SAMPLES:
            wav = np.pad(wav, (0, VAL_SAMPLES - len(wav)))
        else:
            wav = wav[:VAL_SAMPLES]
        wavs.append(torch.from_numpy(wav.astype(np.float32)).unsqueeze(0))
    return wavs


def _predict_from_waveforms(model, mel_transform, wav_list, batch_size=64):
    model.eval()
    preds_clip, preds_fmax, preds_blend = [], [], []
    with torch.no_grad():
        for s in range(0, len(wav_list), batch_size):
            batch = torch.stack(wav_list[s:s+batch_size]).to(device)
            mel = mel_transform(batch)
            B = mel.size(0)
            for i in range(B):
                mel[i] = (mel[i] - mel[i].mean()) / (mel[i].std() + 1e-6)
            mel = mel.to(memory_format=torch.channels_last)
            with autocast():
                clip_logits, framewise = model(mel, return_framewise=True)
                frame_max = framewise.max(dim=1).values
                p_clip = torch.sigmoid(clip_logits).float().cpu().numpy()
                p_fmax = torch.sigmoid(frame_max).float().cpu().numpy()
                p_blend = 0.5 * p_clip + 0.5 * p_fmax
            preds_clip.append(p_clip); preds_fmax.append(p_fmax); preds_blend.append(p_blend)
    return {"clip": np.concatenate(preds_clip),
            "fmax": np.concatenate(preds_fmax),
            "blend": np.concatenate(preds_blend)}

print("OK eval helpers ready")

# ============================================================
# ★ M7-style helpers (per-species AUC, class stats, taxon str)
# ============================================================
def compute_per_species_auc(y_true, y_pred, class_mask=None):
    """Return list of (class_idx, AUC) for non-saturated species."""
    indices = range(y_true.shape[1]) if class_mask is None else class_mask
    aucs = []
    for c in indices:
        col = y_true[:, c]
        if col.sum() == 0 or col.sum() == len(col):
            continue
        try:
            auc = roc_auc_score(col, y_pred[:, c])
            aucs.append((int(c), float(auc)))
        except ValueError:
            continue
    return aucs


def macro_auc_from_list(aucs):
    import numpy as _np_m7
    return float(_np_m7.mean([a for _, a in aucs])) if len(aucs) > 0 else float("nan")


def lowest_k_mean(aucs, k=22):
    import numpy as _np_m7
    if len(aucs) == 0:
        return float("nan")
    sorted_aucs = sorted([a for _, a in aucs])
    k_eff = min(k, len(sorted_aucs))
    return float(_np_m7.mean(sorted_aucs[:k_eff]))


def class_stats_str(aucs):
    import numpy as _np_m7
    if len(aucs) == 0:
        return "n=0 median=nan p25=nan p75=nan #>0.5=0 #>0.7=0 #>0.9=0 #perfect=0"
    vals = _np_m7.array([a for _, a in aucs])
    return (f"n={len(vals)} median={_np_m7.median(vals):.3f} "
            f"p25={_np_m7.percentile(vals,25):.3f} p75={_np_m7.percentile(vals,75):.3f} "
            f"#>0.5={int((vals>0.5).sum())} #>0.7={int((vals>0.7).sum())} "
            f"#>0.9={int((vals>0.9).sum())} #perfect={int((vals>=1.0).sum())}")


def taxon_str_m7(y_true, y_pred, taxon_masks):
    import math as _math_m7
    parts = []
    for t in ["Insecta", "Reptilia", "Amphibia", "Mammalia", "Aves"]:
        mask = taxon_masks.get(t, [])
        if len(mask) == 0:
            parts.append(f"{t}=nan")
            continue
        aucs = compute_per_species_auc(y_true, y_pred, class_mask=list(mask))
        m = macro_auc_from_list(aucs)
        parts.append(f"{t}={m:.3f}" if not _math_m7.isnan(m) else f"{t}=nan")
    return "taxon: " + " ".join(parts)

print("OK M7 helpers ready (compute_per_species_auc, class_stats_str, taxon_str_m7)")


OK eval helpers ready
OK M7 helpers ready (compute_per_species_auc, class_stats_str, taxon_str_m7)


In [9]:
# ============================================================
# Cell 8: build_active_datasets (R2: 3 sources)
# ============================================================
def build_active_datasets(fold_k):
    items = []
    fds = FocalDS(train_df[train_df["fold"] != fold_k],
                  LABEL2IDX, secondary_lookup=focal_secondary_labels, aug=True)
    items.append(("focal", fds, len(fds)))
    if len(sc_meta) > 0:
        vm = sc_meta["fold"].values == fold_k
        sc_train_df = sc_meta[~vm].reset_index(drop=True)
        Y_tr = Y_SC[~vm]
        sds = LabeledSCDS(Y_tr, sc_train_df, aug=True)
        items.append(("labeled_sc", sds, len(sds)))
    # Pseudo SC (not fold-filtered — unlabeled-derived, no leakage)
    pds = PseudoScDS(pseudo_meta, Y_PSEUDO, TS_DIR, aug=True)
    items.append(("pseudo_sc", pds, len(pds)))
    return items

print("OK build_active_datasets ready (focal + labeled_sc + pseudo_sc)")


OK build_active_datasets ready (focal + labeled_sc + pseudo_sc)


In [10]:
# ============================================================
# Cell 9: init — fresh R2 start, optionally resume from Drive
# ============================================================
FOLD_K = FOLDS[0]
print(f"This run trains fold {FOLD_K}")

active = build_active_datasets(FOLD_K)
NAMES, DATASETS, SIZES = zip(*active)
NAMES, DATASETS, SIZES = list(NAMES), list(DATASETS), list(SIZES)
print(f"Streams: {dict(zip(NAMES, SIZES))}")

mds = ConcatDataset(DATASETS)
N_STEPS_PER_EP = max(100, int(sum(SIZES) / BATCH))
print(f"steps/epoch: {N_STEPS_PER_EP}")

model = make_model()
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)
scaler = GradScaler()

warmup_steps = N_STEPS_PER_EP * WARMUP_EPOCHS
total_steps  = N_STEPS_PER_EP * N_TOTAL_EPOCHS
warmup_sched = torch.optim.lr_scheduler.LinearLR(
    optimizer, start_factor=1/25, end_factor=1.0, total_iters=warmup_steps)
cosine_sched = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=total_steps - warmup_steps, eta_min=MIN_LR)
scheduler = torch.optim.lr_scheduler.SequentialLR(
    optimizer, schedulers=[warmup_sched, cosine_sched], milestones=[warmup_steps])

start_epoch = 0
best_ns22 = -1.0
best_macro = -1.0
history = []
resumed = False

# Optional resume from Drive R2
DRIVE_LATEST = DRIVE_OUTPUT_DIR / "ckpt_latest.pth"
if FORCE_FRESH and DRIVE_LATEST.exists():
    print(f"\nFORCE_FRESH=True: ignoring existing R2 ckpt {DRIVE_LATEST}, starting fresh")
elif DRIVE_LATEST.exists():
    print(f"\nFound Drive R2 ckpt: {DRIVE_LATEST}")
    try:
        state = torch.load(str(DRIVE_LATEST), map_location=device, weights_only=False)
    except TypeError:
        state = torch.load(str(DRIVE_LATEST), map_location=device)
    model.load_state_dict(state["model_state"])
    optimizer.load_state_dict(state["optimizer_state"])
    scheduler.load_state_dict(state["scheduler_state"])
    scaler.load_state_dict(state["scaler_state"])
    start_epoch = int(state["epoch"])
    best_ns22 = float(state.get("best_ns22", -1.0))
    best_macro = float(state.get("best_macro", -1.0))
    history = state.get("history", [])
    resumed = True
    print(f"=== RESUMED from epoch {start_epoch}/{N_TOTAL_EPOCHS} ===")
    print(f"    best_ns22={best_ns22:.4f}, best_macro={best_macro:.4f}, history_len={len(history)}")
    src = DRIVE_OUTPUT_DIR / "history.json"
    if src.exists():
        shutil.copy(str(src), str(OUT_DIR / "history.json"))
else:
    print("\nNo Drive R2 ckpt; starting fresh R2")

if start_epoch >= N_TOTAL_EPOCHS:
    print(f"\nAlready trained to epoch {start_epoch}/{N_TOTAL_EPOCHS}.")


This run trains fold 0
Streams: {'focal': 28906, 'labeled_sc': 584, 'pseudo_sc': 31974}
steps/epoch: 640


model.safetensors:   0%|          | 0.00/36.2M [00:00<?, ?B/s]

Backbone out: (1, 512, 8, 9)  (C=512)

No Drive R2 ckpt; starting fresh R2


In [11]:
# ============================================================
# Cell 10: Val prep — load val waveforms once
# ============================================================
if len(sc_meta) > 0:
    vm = sc_meta["fold"].values == FOLD_K
    val_sc_df = sc_meta[vm].reset_index(drop=True)
    Y_val = Y_SC[vm]
    ns22_val = non_s22_mask_sc[vm]
    print(f"Val: {len(val_sc_df)} SS windows for fold {FOLD_K}")
    print(f"  loading val waveforms...")
    _t0 = time.time()
    val_wavs = _load_val_waveforms(val_sc_df)
    print(f"  loaded in {time.time()-_t0:.1f}s")
else:
    val_sc_df = pd.DataFrame()
    Y_val = np.zeros((0, NUM_CLASSES), dtype=np.float32)
    ns22_val = np.zeros(0, dtype=bool)
    val_wavs = []
    print("No labeled SS for validation")


Val: 155 SS windows for fold 0
  loading val waveforms...
  loaded in 0.4s


In [12]:
# ============================================================
# Cell 11: Training loop — Drive mirror per epoch
# ============================================================
mel_transform = MelSpecTransform().to(device)
spec_augment = SpecAugment().to(device)
perch_teacher = PerchTeacher(PERCH_PATH, "cuda" if torch.cuda.is_available() else "cpu") \
                if USE_PERCH_DISTILL else None

def mirror_to_drive(local_path: Path, dst_name: str = None):
    try:
        name = dst_name or local_path.name
        shutil.copy2(str(local_path), str(DRIVE_OUTPUT_DIR / name))
    except Exception as e:
        print(f"  [WARN] mirror_to_drive failed for {local_path.name}: {e}")

epoch_times = []
trained_this_session = 0
ep = start_epoch

for epoch in range(start_epoch, N_TOTAL_EPOCHS):
    elapsed = time.time() - TRAIN_START
    if epoch_times:
        est_next = max(epoch_times[-3:])
        if elapsed + est_next * 1.3 > MAX_RUNTIME_SEC:
            print(f"\n[stop] Time budget exhausted before epoch {epoch+1}/{N_TOTAL_EPOCHS}")
            break

    ep_start = time.time()
    model.train()

    smp = MixSamp(SIZES, NAMES, SHARES, BATCH, N_STEPS_PER_EP, seed=42 + epoch)
    train_loader = DataLoader(
        mds, batch_sampler=smp, collate_fn=collate_m,
        num_workers=NUM_WORKERS,
        persistent_workers=PERSISTENT_WORKERS if NUM_WORKERS > 0 else False,
        pin_memory=True,
        prefetch_factor=4 if NUM_WORKERS > 0 else None,   # 2→4
    )

    el, el_cls, el_dist, nb_count = 0.0, 0.0, 0.0, 0
    for batch_idx, (wav, lb, wt, mk, sr) in enumerate(train_loader):
        wav, lb, wt, mk = wav.to(device, non_blocking=True), lb.to(device, non_blocking=True), \
                          wt.to(device, non_blocking=True), mk.to(device, non_blocking=True)
        sw = mk_sw(sr).to(device, non_blocking=True)

        with torch.no_grad():
            mel = mel_transform(wav)
            B = mel.size(0)
            for i in range(B):
                mel[i] = (mel[i] - mel[i].mean()) / (mel[i].std() + 1e-6)
            mel = spec_augment(mel)
            mel = mel.to(memory_format=torch.channels_last)

        with autocast():
            if USE_PERCH_DISTILL:
                clip_logits, framewise, distill_emb = model(mel, return_framewise=True,
                                                              return_distill=True)
            else:
                clip_logits, framewise = model(mel, return_framewise=True)

            frame_max_logits = framewise.max(dim=1).values
            bce_clip = F.binary_cross_entropy_with_logits(clip_logits, lb, reduction="none")
            bce_frame = F.binary_cross_entropy_with_logits(frame_max_logits, lb, reduction="none")
            bce = 0.5 * bce_clip + 0.5 * bce_frame
            ps = (bce * wt * mk).sum(1) / (mk.sum(1) + 1e-8)
            cls_loss = (ps * sw).mean()

            if USE_PERCH_DISTILL and perch_teacher is not None:
                # ★ exp083 Perch distill: 5s native = direct embed, no averaging
                with torch.no_grad():
                    wav_5s = wav.squeeze(1)
                    N = wav_5s.shape[1]
                    if N > 160000:
                        start_off = (N - 160000) // 2
                        wav_5s = wav_5s[:, start_off:start_off + 160000]
                    elif N < 160000:
                        wav_5s = F.pad(wav_5s, (0, 160000 - N))
                    perch_emb = perch_teacher.embed(wav_5s).to(device)
                distill_loss = F.mse_loss(distill_emb, perch_emb)
                loss = cls_loss + ALPHA_DISTILL * distill_loss
            else:
                distill_loss = torch.tensor(0.0, device=device)
                loss = cls_loss

        optimizer.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        el += float(loss.item())
        el_cls += float(cls_loss.item())
        el_dist += float(distill_loss.item())
        nb_count += 1

        if batch_idx % 20 == 0:
            cur_lr = optimizer.param_groups[0]["lr"]
            print(f"    ep{epoch+1:02d} batch {batch_idx:4d}/{N_STEPS_PER_EP}  "
                  f"loss={loss.item():.4f}  cls={cls_loss.item():.4f}  "
                  f"dist={distill_loss.item():.4f}  lr={cur_lr:.2e}", flush=True)

    train_loss_avg = el / max(nb_count, 1)
    cls_loss_avg = el_cls / max(nb_count, 1)
    dist_loss_avg = el_dist / max(nb_count, 1)

    if len(val_wavs) > 0:
        val_preds_dict = _predict_from_waveforms(model, mel_transform, val_wavs)
        val_preds = val_preds_dict["blend"]
        r = full_eval(Y_val, val_preds, ns22_val, TAXON_MASKS)
        val_metrics = {
            "ns22": r["non_s22_macro"],
            "macro": r["macro_auc_all"],
            "per_taxon": r["per_taxon"],
        }
    else:
        val_metrics = {"ns22": float("nan"), "macro": float("nan"), "per_taxon": {}}

    ep_elapsed = time.time() - ep_start
    epoch_times.append(ep_elapsed)
    trained_this_session += 1
    ep = epoch + 1

    state_to_save = {
        "epoch": ep,
        "fold": FOLD_K,
        "n_total_epochs": N_TOTAL_EPOCHS,
        "model_state": {k: v.cpu() for k, v in model.state_dict().items()},
        "optimizer_state": optimizer.state_dict(),
        "scheduler_state": scheduler.state_dict(),
        "scaler_state": scaler.state_dict(),
        "best_ns22": best_ns22,
        "best_macro": best_macro,
        "history": history,
    }
    torch.save(state_to_save, OUT_DIR / "ckpt_latest.pth")
    torch.save(state_to_save, OUT_DIR / f"ckpt_ep{ep:02d}.pth")

    is_best_ns22 = (not math.isnan(val_metrics["ns22"])) and val_metrics["ns22"] > best_ns22
    is_best_macro = (not math.isnan(val_metrics["macro"])) and val_metrics["macro"] > best_macro
    if is_best_ns22:
        best_ns22 = val_metrics["ns22"]
        state_to_save["best_ns22"] = best_ns22
        torch.save(state_to_save, OUT_DIR / "ckpt_best_ns22.pth")
    if is_best_macro:
        best_macro = val_metrics["macro"]
        state_to_save["best_macro"] = best_macro
        torch.save(state_to_save, OUT_DIR / "ckpt_best_macro.pth")

    cur_lr = optimizer.param_groups[0]["lr"]
    history.append({
        "epoch": ep,
        "train_loss": round(train_loss_avg, 5),
        "cls_loss": round(cls_loss_avg, 5),
        "dist_loss": round(dist_loss_avg, 5),
        "val_ns22": val_metrics["ns22"],
        "val_macro": val_metrics["macro"],
        "val_per_taxon": val_metrics["per_taxon"],
        "lr": cur_lr,
        "elapsed_sec": round(ep_elapsed, 1),
    })
    with open(OUT_DIR / "history.json", "w") as f:
        json.dump(history, f, indent=2, default=str)

    mirror_to_drive(OUT_DIR / "ckpt_latest.pth")
    mirror_to_drive(OUT_DIR / "history.json")
    if is_best_ns22:
        mirror_to_drive(OUT_DIR / "ckpt_best_ns22.pth")
    if is_best_macro:
        mirror_to_drive(OUT_DIR / "ckpt_best_macro.pth")

    total_elapsed = time.time() - TRAIN_START
    # ★ M7-style log: epoch summary + taxon line + class stats + BEST tag
    cur_lr_log = optimizer.param_groups[0]["lr"]
    best_tag = ""
    if is_best_ns22 and is_best_macro:
        best_tag = "BEST(ns22+macro) "
    elif is_best_ns22:
        best_tag = "BEST(ns22) "
    elif is_best_macro:
        best_tag = "BEST(macro) "

    # Per-species AUC for class stats + taxon line (use ns22 mask = non-S22 evaluation set)
    try:
        _val_pred_blend_m7 = val_preds  # already computed above
        _val_true_m7 = Y_val
        if ns22_val is not None and ns22_val.any():
            _val_true_m7_ns22 = _val_true_m7[ns22_val]
            _val_pred_m7_ns22 = _val_pred_blend_m7[ns22_val]
        else:
            _val_true_m7_ns22 = _val_true_m7
            _val_pred_m7_ns22 = _val_pred_blend_m7
        _per_sp_aucs_m7 = compute_per_species_auc(_val_true_m7_ns22, _val_pred_m7_ns22)
        _cls_line_m7 = class_stats_str(_per_sp_aucs_m7)
        _tax_line_m7 = taxon_str_m7(_val_true_m7_ns22, _val_pred_m7_ns22, TAXON_MASKS)
    except Exception as _e_m7:
        _cls_line_m7 = f"(class stats err: {str(_e_m7)[:80]})"
        _tax_line_m7 = f"(taxon err: {str(_e_m7)[:80]})"

    print(f"\n=== Ep {ep}/{N_TOTAL_EPOCHS}: "
          f"loss={train_loss_avg:.4f} (cls={cls_loss_avg:.4f} dist={dist_loss_avg:.4f}) "
          f"val_ns22={val_metrics['ns22']:.4f} val_macro={val_metrics['macro']:.4f} "
          f"{best_tag}lr={cur_lr_log:.2e} "
          f"({ep_elapsed:.0f}s, total {total_elapsed/60:.1f}min) ===")
    print(f"    {_tax_line_m7}")
    print(f"    class: {_cls_line_m7}\n")

    gc.collect()
    torch.cuda.empty_cache()

trained_up_to = ep
print(f"\n=== Session done: trained epochs {start_epoch+1}-{trained_up_to} this session ===")
print(f"    Cumulative: {trained_up_to}/{N_TOTAL_EPOCHS} epochs done")


PerchTeacher: embed_idx=0, providers=['CUDAExecutionProvider', 'CPUExecutionProvider']
    ep01 batch    0/640  loss=0.9923  cls=0.9235  dist=0.0688  lr=1.50e-05
    ep01 batch   20/640  loss=0.8424  cls=0.8186  dist=0.0238  lr=1.87e-05
    ep01 batch   40/640  loss=0.7827  cls=0.7629  dist=0.0198  lr=2.24e-05
    ep01 batch   60/640  loss=0.7341  cls=0.7156  dist=0.0185  lr=2.61e-05
    ep01 batch   80/640  loss=0.6989  cls=0.6807  dist=0.0181  lr=2.98e-05
    ep01 batch  100/640  loss=0.6559  cls=0.6428  dist=0.0131  lr=3.35e-05
    ep01 batch  120/640  loss=0.6177  cls=0.6042  dist=0.0135  lr=3.72e-05
    ep01 batch  140/640  loss=0.5605  cls=0.5480  dist=0.0124  lr=4.09e-05
    ep01 batch  160/640  loss=0.5022  cls=0.4904  dist=0.0118  lr=4.46e-05
    ep01 batch  180/640  loss=0.4358  cls=0.4233  dist=0.0125  lr=4.83e-05
    ep01 batch  200/640  loss=0.3700  cls=0.3588  dist=0.0113  lr=5.20e-05
    ep01 batch  220/640  loss=0.3287  cls=0.3168  dist=0.0119  lr=5.57e-05
    ep01 batc

In [13]:
# ============================================================
# Cell 12: Final Drive mirror — all per-epoch ckpts
# ============================================================
print("Mirroring all artifacts to Drive...")
n_copied = 0
for f in sorted(OUT_DIR.glob("*")):
    if not f.is_file(): continue
    if f.suffix not in {".pth", ".json", ".txt"}: continue
    try:
        shutil.copy2(str(f), str(DRIVE_OUTPUT_DIR / f.name))
        n_copied += 1
        print(f"  {f.name}  {f.stat().st_size/1e6:.1f} MB")
    except Exception as e:
        print(f"  [WARN] {f.name}: {e}")
print(f"\nMirrored {n_copied} files to {DRIVE_OUTPUT_DIR}")


Mirroring all artifacts to Drive...
  ckpt_best_macro.pth  118.0 MB
  ckpt_best_ns22.pth  118.0 MB
  ckpt_ep01.pth  118.0 MB
  ckpt_ep02.pth  118.0 MB
  ckpt_ep03.pth  118.0 MB
  ckpt_ep04.pth  118.0 MB
  ckpt_ep05.pth  118.0 MB
  ckpt_ep06.pth  118.0 MB
  ckpt_ep07.pth  118.0 MB
  ckpt_ep08.pth  118.0 MB
  ckpt_ep09.pth  118.0 MB
  ckpt_ep10.pth  118.0 MB
  ckpt_ep11.pth  118.0 MB
  ckpt_ep12.pth  118.0 MB
  ckpt_ep13.pth  118.0 MB
  ckpt_ep14.pth  118.0 MB
  ckpt_ep15.pth  118.0 MB
  ckpt_ep16.pth  118.0 MB
  ckpt_ep17.pth  118.0 MB
  ckpt_ep18.pth  118.0 MB
  ckpt_ep19.pth  118.0 MB
  ckpt_ep20.pth  118.0 MB
  ckpt_latest.pth  118.0 MB
  history.json  0.0 MB

Mirrored 24 files to /content/drive/MyDrive/kaggle/birdclef2026/output/exp083/r2


In [14]:
# ============================================================
# Cell 13: Session summary (train phase)
# ============================================================
total_time = time.time() - TRAIN_START
print(f"\n{'='*60}")
print(f"R2 Train Session summary")
print(f"{'='*60}")
print(f"  Resume status:        {'RESUMED' if resumed else 'FRESH START'}")
print(f"  Resumed from epoch:   {start_epoch}")
print(f"  Trained this session: {trained_this_session} epoch(s)")
print(f"  Cumulative:           {trained_up_to}/{N_TOTAL_EPOCHS} epochs")
print(f"  Best ns22 AUC:        {best_ns22:.4f}")
print(f"  Best macro AUC:       {best_macro:.4f}")
print(f"  Total session time:   {total_time/60:.1f} min")
print(f"  Drive output:         {DRIVE_OUTPUT_DIR}")
if trained_up_to >= N_TOTAL_EPOCHS:
    print(f"\n  >>> R2 Training COMPLETE — proceed to pseudo phase <<<")
else:
    print(f"\n  ... R2 not yet complete; re-run NB to continue.")
    print(f"  ... Skip pseudo/upload cells until training completes.")



R2 Train Session summary
  Resume status:        FRESH START
  Resumed from epoch:   0
  Trained this session: 20 epoch(s)
  Cumulative:           20/20 epochs
  Best ns22 AUC:        0.9162
  Best macro AUC:       0.9162
  Total session time:   130.6 min
  Drive output:         /content/drive/MyDrive/kaggle/birdclef2026/output/exp083/r2

  >>> R2 Training COMPLETE — proceed to pseudo phase <<<


## ─────── R2 Pseudo Phase ───────

R2 学生の best ckpt で全 train_soundscapes 推論 → R2 pseudo CSV を Drive に保存。
exp017 R3 等で使う想定。


In [15]:
# ============================================================
# Cell 15: Pseudo setup — free VRAM, load best ckpt
# ============================================================
import glob
from scipy.ndimage import gaussian_filter1d

# Post-processing config (R1 と同じ)
N_WINDOWS    = 12   # exp083: 60s / 5s = 12
CHUNK_N      = TRAIN_SAMPLES
GAUSS_SIGMA  = 0.65
POWER_GAMMA  = 1.2

# Free VRAM from training
del optimizer, scheduler, scaler, mel_transform, spec_augment
if perch_teacher is not None:
    del perch_teacher
del model
gc.collect()
torch.cuda.empty_cache()

# Best ckpt path
CKPT_PATH = OUT_DIR / "ckpt_best_ns22.pth"
if not CKPT_PATH.exists():
    CKPT_PATH = OUT_DIR / "ckpt_latest.pth"
assert CKPT_PATH.exists(), f"No ckpt at {OUT_DIR}"
print(f"Using ckpt: {CKPT_PATH}")
print(f"R2 pseudo dir: {DRIVE_R2_PSEUDO_DIR}")

try:
    state = torch.load(str(CKPT_PATH), map_location="cpu", weights_only=False)
except TypeError:
    state = torch.load(str(CKPT_PATH), map_location="cpu")

print(f"  epoch={state.get('epoch')}, "
      f"best_ns22={state.get('best_ns22', float('nan')):.4f}, "
      f"best_macro={state.get('best_macro', float('nan')):.4f}")

model = BirdSEDModel().to(device)
model.load_state_dict(state["model_state"], strict=False)
model.eval()
model = model.to(memory_format=torch.channels_last)
print(f"OK model loaded ({sum(p.numel() for p in model.parameters())/1e6:.1f}M params)")

mel_tf = MelSpecTransform().to(device)


Using ckpt: /content/output/ckpt_best_ns22.pth
R2 pseudo dir: /content/drive/MyDrive/kaggle/birdclef2026/output/exp083/r2-pseudo
  epoch=18, best_ns22=0.9162, best_macro=0.9152
Backbone out: (1, 512, 8, 9)  (C=512)
OK model loaded (9.8M params)


In [16]:
# ============================================================
# Cell 16: Pseudo R2 inference — all train_soundscapes
# ============================================================
import soundfile as sf
import librosa
from tqdm.auto import tqdm

def load_audio_32k_mono(path, target_samples=60 * SR):
    wav, sr = sf.read(str(path), dtype="float32", always_2d=False)
    if wav.ndim > 1:
        wav = wav.mean(axis=1)
    if sr != SR:
        wav = librosa.resample(wav, orig_sr=sr, target_sr=SR)
    if len(wav) < target_samples:
        wav = np.pad(wav, (0, target_samples - len(wav)))
    elif len(wav) > target_samples:
        wav = wav[:target_samples]
    return wav.astype(np.float32)

def file_to_chunks(path):
    wav = load_audio_32k_mono(path, target_samples=N_WINDOWS * CHUNK_N)
    return wav.reshape(N_WINDOWS, CHUNK_N).astype(np.float32)

sc_files = sorted(glob.glob(str(TS_DIR / "*.ogg")))
print(f"train_soundscapes: {len(sc_files)} files")
assert len(sc_files) > 0

all_filenames, all_start_secs, all_end_secs, all_probs = [], [], [], []
t0 = time.time()

with torch.no_grad():
    for fi, fpath in enumerate(tqdm(sc_files, desc="infer", mininterval=2.0)):
        stem = Path(fpath).stem
        try:
            chunks = file_to_chunks(fpath)
        except Exception as e:
            print(f"WARN: {stem}: {e}")
            chunks = np.zeros((N_WINDOWS, CHUNK_N), dtype=np.float32)

        wav_t = torch.from_numpy(chunks).unsqueeze(1).to(device)
        mel = mel_tf(wav_t)
        for i in range(mel.size(0)):
            mel[i] = (mel[i] - mel[i].mean()) / (mel[i].std() + 1e-6)
        mel = mel.to(memory_format=torch.channels_last)

        with autocast():
            clip_logits, framewise = model(mel, return_framewise=True)
            frame_max = framewise.max(dim=1).values
            p_clip = torch.sigmoid(clip_logits).float().cpu().numpy()
            p_fmax = torch.sigmoid(frame_max).float().cpu().numpy()
        probs_file = 0.5 * p_clip + 0.5 * p_fmax
        probs_file = gaussian_filter1d(probs_file, sigma=GAUSS_SIGMA, axis=0,
                                        mode="nearest").astype(np.float32)

        all_probs.append(probs_file)
        for wi in range(N_WINDOWS):
            all_filenames.append(stem)
            all_start_secs.append(wi * TRAIN_DURATION)
            all_end_secs.append((wi + 1) * TRAIN_DURATION)

prob_mat = np.concatenate(all_probs, axis=0).astype(np.float32)
filenames_arr = np.array(all_filenames)
start_secs_arr = np.array(all_start_secs, dtype=np.float32)
end_secs_arr   = np.array(all_end_secs,   dtype=np.float32)
print(f"\nInference: {prob_mat.shape}, mean={prob_mat.mean():.4f}, max={prob_mat.max():.4f} "
      f"in {(time.time()-t0)/60:.1f} min")


train_soundscapes: 10658 files


infer:   0%|          | 0/10658 [00:00<?, ?it/s]


Inference: (127896, 234), mean=0.0073, max=0.9961 in 5.8 min


In [17]:
# ============================================================
# Cell 17: Pseudo R2 postproc + save CSV + Drive mirror
# ============================================================
print(f"Pre-PT: mean={prob_mat.mean():.6f}, max={prob_mat.max():.4f}, "
      f"99%ile={np.percentile(prob_mat, 99):.4f}")

prob_mat = np.power(prob_mat, POWER_GAMMA).astype(np.float32)

print(f"Post-PT (γ={POWER_GAMMA}): mean={prob_mat.mean():.6f}, max={prob_mat.max():.4f}, "
      f"99%ile={np.percentile(prob_mat, 99):.4f}, "
      f"50%ile={np.percentile(prob_mat, 50):.4f}")

row_max = prob_mat.max(axis=1)
print(f"row_max stats (diagnostic, NO filter): 50%ile={np.percentile(row_max,50):.4f}, "
      f"99%ile={np.percentile(row_max,99):.4f}")

df = pd.DataFrame(prob_mat, columns=PRIMARY_LABELS)
df.insert(0, "filename",  filenames_arr)
df.insert(1, "start_sec", start_secs_arr)
df.insert(2, "end_sec",   end_secs_arr)

local_csv = OUT_DIR / "pseudo_labels.csv"
df.to_csv(local_csv, index=False)
print(f"\nLocal: {local_csv} ({local_csv.stat().st_size/1024/1024:.1f}MB)")
print(f"  shape: {df.shape}, files: {df['filename'].nunique()}")

# Mirror to Drive
drive_csv = DRIVE_R2_PSEUDO_DIR / "pseudo_labels.csv"
sz = local_csv.stat().st_size
print(f"\nMirroring to Drive: {drive_csv} ({sz/1e6:.1f}MB)")
t0 = time.time()
with open(local_csv, "rb") as fin, open(drive_csv, "wb") as fout:
    pbar = tqdm(total=sz, unit="B", unit_scale=True, unit_divisor=1024,
                desc="Drive mirror", mininterval=1.0)
    while True:
        buf = fin.read(8*1024*1024)
        if not buf: break
        fout.write(buf); pbar.update(len(buf))
    pbar.close()
print(f"  done in {time.time()-t0:.0f}s")

meta = {
    "backbone": BACKBONE,
    "round": "R2",
    "ckpt": str(CKPT_PATH.name),
    "epoch": state.get("epoch"),
    "best_ns22": state.get("best_ns22"),
    "best_macro": state.get("best_macro"),
    "n_files": int(df["filename"].nunique()),
    "n_rows": int(len(df)),
    "gauss_sigma": GAUSS_SIGMA,
    "power_gamma": POWER_GAMMA,
}
(DRIVE_R2_PSEUDO_DIR / "pseudo_meta.json").write_text(json.dumps(meta, indent=2, default=str))
print(f"  meta written: {DRIVE_R2_PSEUDO_DIR}/pseudo_meta.json")

# Free pseudo memory before upload
del prob_mat, df, all_probs, all_filenames
gc.collect(); torch.cuda.empty_cache()


Pre-PT: mean=0.007350, max=0.9961, 99%ile=0.1592
Post-PT (γ=1.2): mean=0.005628, max=0.9954, 99%ile=0.1102, 50%ile=0.0001
row_max stats (diagnostic, NO filter): 50%ile=0.4786, 99%ile=0.9781

Local: /content/output/pseudo_labels.csv (391.2MB)
  shape: (127896, 237), files: 10658

Mirroring to Drive: /content/drive/MyDrive/kaggle/birdclef2026/output/exp083/r2-pseudo/pseudo_labels.csv (410.2MB)


Drive mirror:   0%|          | 0.00/391M [00:00<?, ?B/s]

  done in 1s
  meta written: /content/drive/MyDrive/kaggle/birdclef2026/output/exp083/r2-pseudo/pseudo_meta.json


In [18]:
# ============================================================
# Cell: Upload R2 weights to Kaggle Dataset (R1 from Drive + R2 from local)
# ============================================================
import tempfile, time
from kaggle.api.kaggle_api_extended import KaggleApi

# ★ Re-authenticate at upload start
print("Re-authenticating Kaggle API for upload...")
KAGGLE_CFG = Path.home() / ".kaggle" / "kaggle.json"
if KAGGLE_CFG.exists():
    creds = json.loads(KAGGLE_CFG.read_text())
    if creds.get("key", "").startswith("KGAT_"):
        os.environ["KAGGLE_API_TOKEN"] = creds["key"]
api = KaggleApi(); api.authenticate()
print("  Re-auth OK")

USER  = "maekeso"
SLUG  = "birdclef2026-exp083-weights"
TITLE = "birdclef2026 exp083 weights"

# R1 files location (Drive) — sibling of DRIVE_OUTPUT_DIR (R2 dir)
DRIVE_R1_DIR = DRIVE_OUTPUT_DIR.parent / "r1"

# R1 ckpts to copy from Drive (R1 stage)
TARGETS_R1 = [
    "ckpt_best_ns22.pth",
    "ckpt_best_macro.pth",
    "ckpt_latest.pth",
    "history.json",
]
# R2 ckpts to copy from local OUT_DIR (R2 stage)
TARGETS_R2 = [
    "ckpt_best_ns22.pth",
    "ckpt_best_macro.pth",
    "ckpt_latest.pth",
    "history.json",
]

print(f"\nUpload target: {USER}/{SLUG}")
print(f"  R1 source: {DRIVE_R1_DIR}")
print(f"  R2 source: {OUT_DIR}")

with tempfile.TemporaryDirectory() as td:
    td = Path(td)

    # ============================================================
    # [1/3] Stage R1 files (Drive → temp_dir/r1/)
    # ============================================================
    r1_dst = td / "r1"
    r1_dst.mkdir(parents=True, exist_ok=True)
    print(f"\n[1/3] Staging R1 files (Drive → temp/r1/)")
    n_r1 = 0
    r1_mb = 0.0
    if not DRIVE_R1_DIR.exists():
        print(f"  [WARN] DRIVE_R1_DIR not found: {DRIVE_R1_DIR}")
        print(f"  → R1 will NOT be preserved. R2 upload will proceed.")
    else:
        for fn in TARGETS_R1:
            src = DRIVE_R1_DIR / fn
            if not src.exists():
                print(f"  skip (missing on Drive): r1/{fn}")
                continue
            shutil.copy2(str(src), str(r1_dst / fn))
            sz_mb = src.stat().st_size / 1e6
            r1_mb += sz_mb
            n_r1 += 1
            print(f"  R1 staged: r1/{fn}  ({sz_mb:.1f}MB)")
    print(f"  R1 TOTAL: {n_r1} files, {r1_mb:.1f}MB")

    # ============================================================
    # [2/3] Stage R2 files (local OUT_DIR → temp_dir/r2/)
    # ============================================================
    r2_dst = td / "r2"
    r2_dst.mkdir(parents=True, exist_ok=True)
    print(f"\n[2/3] Staging R2 files (local OUT_DIR → temp/r2/)")
    n_r2 = 0
    r2_mb = 0.0
    for fn in TARGETS_R2:
        src = OUT_DIR / fn
        if not src.exists():
            print(f"  skip (missing local): r2/{fn}")
            continue
        shutil.copy2(str(src), str(r2_dst / fn))
        sz_mb = src.stat().st_size / 1e6
        r2_mb += sz_mb
        n_r2 += 1
        print(f"  R2 staged: r2/{fn}  ({sz_mb:.1f}MB)")
    assert n_r2 > 0, "No R2 files to upload! Aborting."
    print(f"  R2 TOTAL: {n_r2} files, {r2_mb:.1f}MB")

    # Final staging summary
    print(f"\n  Final staging tree:")
    for sub in ["r1", "r2"]:
        sub_dir = td / sub
        if sub_dir.exists():
            for f in sorted(sub_dir.iterdir()):
                if f.is_file():
                    sz_mb = f.stat().st_size / 1e6
                    print(f"    {sub}/{f.name}  ({sz_mb:.1f}MB)")
    total_mb = sum(f.stat().st_size for f in td.rglob("*") if f.is_file()) / 1e6
    print(f"  Grand total: {total_mb:.1f}MB")

    # ============================================================
    # [3/3] Upload (create_version → fallback create_new)
    # ============================================================
    meta = {
        "title": TITLE,
        "id": f"{USER}/{SLUG}",
        "licenses": [{"name": "CC0-1.0"}],
    }
    (td / "dataset-metadata.json").write_text(json.dumps(meta, indent=2), encoding="utf-8")

    version_notes = f"R2 best_ns22={best_ns22:.4f}, best_macro={best_macro:.4f} (r1/ + r2/ folders)"
    uploaded = False
    last_err = None

    print(f"\n[3/3] Uploading to Kaggle Dataset...")
    print(f"  version_notes: {version_notes}")
    try:
        t0 = time.time()
        api.dataset_create_version(folder=str(td),
                                    version_notes=version_notes,
                                    dir_mode="zip", quiet=False)
        print(f"\n  OK Uploaded (new version, {time.time()-t0:.0f}s)")
        uploaded = True
    except Exception as e:
        msg = str(e)[:600]
        print(f"  create_version FAILED: type={type(e).__name__}")
        print(f"    msg: {msg}")
        last_err = e
        if "not found" in msg.lower() or "404" in msg or "Could not find" in msg:
            print(f"\n  Dataset doesn\'t exist → try create_new...")
            try:
                t0 = time.time()
                api.dataset_create_new(folder=str(td), public=False,
                                       dir_mode="zip", quiet=False)
                print(f"  OK Created (first time, {time.time()-t0:.0f}s)")
                uploaded = True
            except Exception as e2:
                msg2 = str(e2)[:600]
                print(f"  create_new FAILED: type={type(e2).__name__}")
                print(f"    msg: {msg2}")
                last_err = e2

    if uploaded:
        print(f"\n[OK] Dataset URL: https://www.kaggle.com/datasets/{USER}/{SLUG}")
        print(f"     Structure: r1/{{ckpt_*.pth, history.json}}, r2/{{ckpt_*.pth, history.json}}")
        print(f"     Both R1 + R2 visible in latest version")
    else:
        print(f"\n[FAIL] Upload failed. Last error:")
        print(f"  {type(last_err).__name__}: {str(last_err)[:500]}")
        print(f"\nFiles staged at {td} (will be cleaned up).")


Re-authenticating Kaggle API for upload...
  Re-auth OK

Upload target: maekeso/birdclef2026-exp083-weights
  R1 source: /content/drive/MyDrive/kaggle/birdclef2026/output/exp083/r1
  R2 source: /content/output

[1/3] Staging R1 files (Drive → temp/r1/)
  R1 staged: r1/ckpt_best_ns22.pth  (118.0MB)
  R1 staged: r1/ckpt_best_macro.pth  (118.0MB)
  R1 staged: r1/ckpt_latest.pth  (118.0MB)
  R1 staged: r1/history.json  (0.0MB)
  R1 TOTAL: 4 files, 354.1MB

[2/3] Staging R2 files (local OUT_DIR → temp/r2/)
  R2 staged: r2/ckpt_best_ns22.pth  (118.0MB)
  R2 staged: r2/ckpt_best_macro.pth  (118.0MB)
  R2 staged: r2/ckpt_latest.pth  (118.0MB)
  R2 staged: r2/history.json  (0.0MB)
  R2 TOTAL: 4 files, 354.1MB

  Final staging tree:
    r1/ckpt_best_macro.pth  (118.0MB)
    r1/ckpt_best_ns22.pth  (118.0MB)
    r1/ckpt_latest.pth  (118.0MB)
    r1/history.json  (0.0MB)
    r2/ckpt_best_macro.pth  (118.0MB)
    r2/ckpt_best_ns22.pth  (118.0MB)
    r2/ckpt_latest.pth  (118.0MB)
    r2/history.json 

100%|██████████| 311M/311M [00:08<00:00, 38.0MB/s]


Upload successful: r2.zip (311MB)
Starting upload for file r1.zip


100%|██████████| 311M/311M [00:08<00:00, 37.7MB/s]


Upload successful: r1.zip (311MB)

  OK Uploaded (new version, 40s)

[OK] Dataset URL: https://www.kaggle.com/datasets/maekeso/birdclef2026-exp083-weights
     Structure: r1/{ckpt_*.pth, history.json}, r2/{ckpt_*.pth, history.json}
     Both R1 + R2 visible in latest version


In [19]:
# ============================================================
# Cell 19: Terminate Colab runtime — save Pro+ compute units
# ============================================================
# ここまで完走したら Blackwell を即座に切る。Drive 出力済 + Kaggle Dataset push 済で残作業なし。
print("All R2 phases complete. Terminating Colab runtime in 5s to free compute units...")
import time as _t
_t.sleep(5)
from google.colab import runtime
runtime.unassign()


All R2 phases complete. Terminating Colab runtime in 5s to free compute units...
